# CISEI Scenario Planner

Notebook de trabalho para o fluxo integrado de criação de cenários CISEI.

### Fluxo desta versão

1. Definir o projeto.
2. Inicializar `ScenarioAgentOpenAI` + plugin CISEI.
3. Conversar com o agente.
4. Validar o cenário.
5. Confirmar e gerar/salvar `scenario.toml`.
6. Ler `nodes.csv` do mesmo projeto.
7. Visualizar os nós com `ipyleaflet`.

> **Nesta versão não executamos Planning Service, Network Service, SDK ou MCP.**
> O objetivo é validar a camada `conversa → ScenarioBuilder → TOML → CSV → mapa`.


## 1. CONFIGURAÇÃO DO PROJETO

O nome do projeto é definido explicitamente pelo notebook. O LLM não escolhe nem inventa o nome.

Todos os artefatos do projeto ficam em:

`/home/jovyan/work/cisei_workspace/projects/<PROJECT_NAME>/`


In [23]:
from pathlib import Path

# ============================================================
# PROJETO
# ============================================================

PROJECT_NAME = "curitiba_lte"

WORKSPACE_ROOT = Path("/home/jovyan/work/cisei_workspace")
PROJECTS_ROOT = WORKSPACE_ROOT / "projects"
PROJECT_ROOT = PROJECTS_ROOT / PROJECT_NAME

SCENARIO_PATH = PROJECT_ROOT / "scenario.toml"
NODES_PATH = PROJECT_ROOT / "nodes.csv"
RESULTS_DIR = PROJECT_ROOT / "results"
REPORTS_DIR = PROJECT_ROOT / "reports"

# Criar somente a estrutura de diretórios.
# Não criar scenario.toml nem nodes.csv automaticamente.

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 72)
print("CONFIGURAÇÃO DO PROJETO")
print("=" * 72)
print("PROJECT_NAME :", PROJECT_NAME)
print("PROJECT_ROOT :", PROJECT_ROOT)
print("SCENARIO_PATH:", SCENARIO_PATH)
print("NODES_PATH   :", NODES_PATH)
print("RESULTS_DIR  :", RESULTS_DIR)
print("REPORTS_DIR  :", REPORTS_DIR)
print("=" * 72)

CONFIGURAÇÃO DO PROJETO
PROJECT_NAME : curitiba_lte
PROJECT_ROOT : /home/jovyan/work/cisei_workspace/projects/curitiba_lte
SCENARIO_PATH: /home/jovyan/work/cisei_workspace/projects/curitiba_lte/scenario.toml
NODES_PATH   : /home/jovyan/work/cisei_workspace/projects/curitiba_lte/nodes.csv
RESULTS_DIR  : /home/jovyan/work/cisei_workspace/projects/curitiba_lte/results
REPORTS_DIR  : /home/jovyan/work/cisei_workspace/projects/curitiba_lte/reports


## 2. AGENTE

Usamos o agente OpenAI já validado e o provider do plugin CISEI.

O plugin fornece **contexto de domínio**. Ele não executa planejamento, não executa MCP e não fornece defaults de cenário.

O `ScenarioBuilder` continua sendo a autoridade para o estado, validação e geração do TOML.


In [24]:
import json
import sys

PROJECT_CODE_ROOT = Path("/home/jovyan/work/planapp-mcp")
PLUGIN_ROOT = PROJECT_CODE_ROOT / "cisei-planning-engineering"

if str(PROJECT_CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_CODE_ROOT))

from scenario_agent_openai import create_scenario_agent_openai
from scenario_domain.providers import CISEIPluginDomainContextProvider


# ============================================================
# CONTEXTO DE DOMÍNIO DO PLUGIN
# ============================================================

DOMAIN_CONTEXT_SECTIONS = (
    "scenario_format",
    "assistant_workflow",
    "design_gaps_and_decisions",
)

provider = CISEIPluginDomainContextProvider(
    PLUGIN_ROOT
)


# ============================================================
# AGENTE
# ============================================================

agent = create_scenario_agent_openai(
    PROJECT_ROOT,
    domain_context_provider=provider,
    domain_context_sections=DOMAIN_CONTEXT_SECTIONS,
)


print("=" * 72)
print("AGENTE INICIALIZADO")
print("=" * 72)
print("PROJECT_CODE_ROOT:", PROJECT_CODE_ROOT)
print("PLUGIN_ROOT      :", PLUGIN_ROOT)
print("PROJECT_ROOT     :", PROJECT_ROOT)
print("AGENT PROJECT    :", agent.project_path)
print("PLUGIN PROVIDER  :", provider.provider_id)
print("PLUGIN VERSION   :", provider.provider_version)
print("DOMAIN SECTIONS  :", agent.domain_context_sections)
print("=" * 72)


AGENTE INICIALIZADO
PROJECT_CODE_ROOT: /home/jovyan/work/planapp-mcp
PLUGIN_ROOT      : /home/jovyan/work/planapp-mcp/cisei-planning-engineering
PROJECT_ROOT     : /home/jovyan/work/cisei_workspace/projects/curitiba_lte
AGENT PROJECT    : /home/jovyan/work/cisei_workspace/projects/curitiba_lte
PLUGIN PROVIDER  : cisei_plugin
PLUGIN VERSION   : 0.4.0
DOMAIN SECTIONS  : ('scenario_format', 'assistant_workflow', 'design_gaps_and_decisions')


### Verificação opcional do contexto do plugin

Mantemos a seleção de seções definida para o agente. Não carregamos automaticamente as 11 referências do plugin.


In [25]:
print("Seções de contexto do agente:")

for section in agent.domain_context_sections:
    print(" -", section)

print()
print("Capabilities do provider:")

for key, value in provider.capabilities().items():
    print(f" - {key}: {value}")


Seções de contexto do agente:
 - scenario_format
 - assistant_workflow
 - design_gaps_and_decisions

Capabilities do provider:
 - domain_context: True
 - section_selection: True
 - provenance: True
 - versioning: True
 - scenario_defaults: False
 - scenario_mutation: False
 - scenario_validation: False
 - planning_execution: False
 - mcp_execution: False
 - llm_execution: False


## 3. CONVERSAÇÃO INTERATIVA

Esta é a célula principal de trabalho.

**Nenhuma mensagem é enviada automaticamente.**

Digite uma mensagem no campo abaixo e clique em **Enviar**.

O agente mantém o histórico incremental da conversa e o `ScenarioBuilder` mantém o estado do cenário.

O painel mostra:

* mensagem enviada pelo usuário;
* resposta estruturada do agente;
* estado atual do cenário;
* `valid`;
* `complete`;
* `confirmed`;
* `missing`;
* `next_action`.



In [26]:
import html
import json

import ipywidgets as widgets
from IPython.display import display


# ============================================================
# FUNÇÕES DE ESTADO
# ============================================================

def obter_estado():
    return agent.scenario_agent.conversation_state()


def renderizar_estado():
    estado = obter_estado()

    estado_html = f"""
    <div style="
        font-family: monospace;
        line-height: 1.6;
        padding: 8px;
    ">
        <b>ESTADO DO CENÁRIO</b><br>
        valid: <b>{html.escape(str(estado.get("valid")))}</b><br>
        complete: <b>{html.escape(str(estado.get("complete")))}</b><br>
        confirmed: <b>{html.escape(str(estado.get("confirmed")))}</b><br>
        next_action: <b>{html.escape(str(estado.get("next_action")))}</b><br>
        missing: {html.escape(str(estado.get("missing")))}<br>
        errors: {html.escape(str(estado.get("errors")))}
    </div>
    """

    state_output.clear_output(wait=True)

    with state_output:
        display(
            widgets.HTML(
                value=estado_html
            )
        )

    return estado


# ============================================================
# CONVERSAÇÃO
# ============================================================

def conversar(mensagem: str):
    """
    Envia uma nova mensagem ao ScenarioAgentOpenAI.

    Esta função não cria estado artificial e não executa
    nenhuma mensagem automaticamente.
    """

    if not isinstance(mensagem, str):
        raise TypeError("A mensagem deve ser uma string.")

    mensagem = mensagem.strip()

    if not mensagem:
        raise ValueError(
            "A mensagem do usuário não pode ser vazia."
        )

    return agent.chat(mensagem)


# ============================================================
# CALLBACK DO BOTÃO ENVIAR
# ============================================================

def enviar_mensagem(_=None):

    mensagem = input_box.value.strip()

    if not mensagem:
        return

    # Limpa o campo imediatamente.
    input_box.value = ""

    # --------------------------------------------------------
    # HISTÓRICO
    # --------------------------------------------------------

    with conversation_output:

        display(
            widgets.HTML(
                value=(
                    "<hr>"
                    "<b>Usuário</b><br>"
                    f"{html.escape(mensagem)}"
                )
            )
        )

        # ----------------------------------------------------
        # CHAMADA AO AGENTE
        # ----------------------------------------------------

        try:

            resultado = conversar(mensagem)

            resposta = json.dumps(
                resultado,
                ensure_ascii=False,
                indent=2,
                default=str,
            )

            display(
                widgets.HTML(
                    value=(
                        "<b>Agente</b>"
                        "<pre style="
                        "'white-space: pre-wrap;'>"
                        f"{html.escape(resposta)}"
                        "</pre>"
                    )
                )
            )

        except Exception as exc:

            erro = (
                f"{type(exc).__name__}: {exc}"
            )

            display(
                widgets.HTML(
                    value=(
                        "<b>Erro</b>"
                        "<pre style="
                        "'white-space: pre-wrap;'>"
                        f"{html.escape(erro)}"
                        "</pre>"
                    )
                )
            )

    # Atualiza o estado após a mensagem.
    renderizar_estado()


# ============================================================
# WIDGETS
# ============================================================

input_box = widgets.Textarea(
    placeholder=(
        "Digite aqui a próxima mensagem "
        "para o agente..."
    ),
    layout=widgets.Layout(
        width="100%",
        height="90px",
    ),
)

send_button = widgets.Button(
    description="Enviar",
    button_style="primary",
    icon="paper-plane",
)

clear_button = widgets.Button(
    description="Limpar painel",
    icon="trash",
)


# ============================================================
# LIMPAR PAINEL
# ============================================================

def limpar_painel(_=None):

    conversation_output.clear_output()

    renderizar_estado()


# ============================================================
# EVENTOS
# ============================================================

send_button.on_click(
    enviar_mensagem
)

clear_button.on_click(
    limpar_painel
)


# ============================================================
# ÁREAS DE SAÍDA
# ============================================================

conversation_output = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #cccccc",
        padding="10px",
        max_height="500px",
        overflow="auto",
    )
)

state_output = widgets.Output()


# ============================================================
# INTERFACE
# ============================================================

conversation_ui = widgets.VBox(
    [
        input_box,
        widgets.HBox(
            [
                send_button,
                clear_button,
            ]
        ),
        state_output,
        conversation_output,
    ]
)


# ============================================================
# EXIBIR
# ============================================================

display(conversation_ui)

# Apenas mostra o estado inicial.
# NÃO envia nenhuma mensagem ao agente.
renderizar_estado()


{'valid': True,
 'complete': False,
 'confirmed': False,
 'missing': ['scenario.working_crs',
  'instances.source',
  'instances.format',
  'instances.profile_column',
  'antennas.<at_least_one>',
  'interfaces.<at_least_one>',
  'devices.<at_least_one>',
  'metrics.<at_least_one>'],
 'errors': [],
 'next_action': 'collect_missing_fields'}

### Exemplos de mensagens

As mensagens abaixo são **somente exemplos** e não são executadas automaticamente.

Podemos começar, por exemplo, com:

```text
O CRS de trabalho é EPSG:31982.
```

Depois:

```text
Crie uma interface LTE em 915 MHz.
```

Depois:

```text
A potência de transmissão é 43 dBm.
```

E continuar preenchendo somente os valores explicitamente informados.

Quando o cenário estiver completo, o agente deverá indicar que a próxima ação é `confirm`.


## 4. ESTADO, CONFIRMAÇÃO E GERAÇÃO DO `scenario.toml`

O estado é mantido pelo `ScenarioBuilder`.

A sequência esperada é:

```text
conversa
   ↓
update_scenario
   ↓
valid / complete
   ↓
confirm
   ↓
confirmed
   ↓
generate_toml()
   ↓
scenario.toml
```

A confirmação deve ser uma decisão explícita do usuário.



In [39]:
# ============================================================
# ESTADO ATUAL DO CENÁRIO
# ============================================================

def mostrar_estado():
    estado = obter_estado()

    print("=" * 72)
    print("ESTADO ATUAL DO CENÁRIO")
    print("=" * 72)

    print("valid      :", estado.get("valid"))
    print("complete   :", estado.get("complete"))
    print("confirmed  :", estado.get("confirmed"))
    print("next_action:", estado.get("next_action"))
    print("missing    :", estado.get("missing"))
    print("errors     :", estado.get("errors"))

    print()
    print("ESTADO COMPLETO:")
    print(
        json.dumps(
            estado,
            ensure_ascii=False,
            indent=2,
            default=str,
        )
    )

    print("=" * 72)

    return estado


# ------------------------------------------------------------
# CONFIRMAÇÃO
# ------------------------------------------------------------

def verificar_pronto_para_confirmacao():

    estado = obter_estado()

    print("=" * 72)
    print("VERIFICAÇÃO PARA CONFIRMAÇÃO")
    print("=" * 72)

    print("valid      :", estado.get("valid"))
    print("complete   :", estado.get("complete"))
    print("confirmed  :", estado.get("confirmed"))
    print("next_action:", estado.get("next_action"))
    print("missing    :", estado.get("missing"))
    print("errors     :", estado.get("errors"))

    print("=" * 72)

    return estado


def confirmar_cenario():

    estado = obter_estado()

    if not estado.get("valid"):
        raise RuntimeError(
            "O cenário não está válido."
        )

    if not estado.get("complete"):
        raise RuntimeError(
            "O cenário ainda está incompleto."
        )

    if estado.get("confirmed"):
        print("O cenário já está confirmado.")
        return estado

    print("Enviando confirmação ao agente...")

    resultado = conversar(
        "Confirmo o cenário."
    )

    renderizar_estado()

    return resultado


# ============================================================
# EXECUÇÃO DA CÉLULA
# ============================================================

# Mostrar imediatamente o estado atual.
mostrar_estado()

ESTADO ATUAL DO CENÁRIO
valid      : True
complete   : True
confirmed  : True
next_action: finalize
missing    : []
errors     : []

ESTADO COMPLETO:
{
  "valid": true,
  "complete": true,
  "confirmed": true,
  "missing": [],
  "errors": [],
  "next_action": "finalize"
}


{'valid': True,
 'complete': True,
 'confirmed': True,
 'missing': [],
 'errors': [],
 'next_action': 'finalize'}

### Mensagens trocadas com o modelo

Útil para auditoria durante os testes. A lista contém o histórico conversacional do agente.


In [41]:
# ============================================================
# AUDITORIA DA CONVERSA
# ============================================================

def mostrar_mensagens():
    print("=" * 72)
    print("HISTÓRICO DA CONVERSA")
    print("=" * 72)

    mensagens = getattr(agent, "messages", None)

    if not mensagens:
        print("Nenhuma mensagem registrada ainda.")
        print("=" * 72)
        return []

    for i, mensagem in enumerate(mensagens, start=1):
        print()
        print(f"[{i}]")

        if isinstance(mensagem, dict):
            role = mensagem.get("role", "?")
            content = mensagem.get("content", "")

            print("role   :", role)
            print("content:")

            if isinstance(content, str):
                print(content)
            else:
                print(
                    json.dumps(
                        content,
                        ensure_ascii=False,
                        indent=2,
                        default=str,
                    )
                )
        else:
            print(mensagem)

    print()
    print("=" * 72)

    return mensagens


# ============================================================
# EXECUÇÃO DA CÉLULA
# ============================================================

mostrar_mensagens()

HISTÓRICO DA CONVERSA

[1]
role   : user
content:
USER MESSAGE

O CRS de trabalho é EPSG:31982.

APPLICATION CONTEXT

{
  "scenario_state": {
    "valid": true,
    "complete": false,
    "confirmed": false,
    "missing": [
      "scenario.working_crs",
      "instances.source",
      "instances.format",
      "instances.profile_column",
      "antennas.<at_least_one>",
      "interfaces.<at_least_one>",
      "devices.<at_least_one>",
      "metrics.<at_least_one>"
    ],
    "errors": [],
    "next_action": "collect_missing_fields"
  },
  "scenario_snapshot": {
    "scenario": {},
    "antennas": {},
    "interfaces": {},
    "devices": {},
    "metrics": {},
    "instances": {}
  }
}

[2]
role   : assistant
content:
{"action":"update_scenario","updates":{"scenario":{"working_crs":"EPSG:31982"}}}

[3]
role   : user
content:
USER MESSAGE

Crie a antena lte_sector_0. Ela é do tipo sector, modelo commscope_rv_65s_fvb, com descrição "CommScope RV-65S-FVB sector at 0 deg", altura de 7.0 

[{'role': 'user',
  'content': 'USER MESSAGE\n============\n\nO CRS de trabalho é EPSG:31982.\n\nAPPLICATION CONTEXT\n===================\n\n{\n  "scenario_state": {\n    "valid": true,\n    "complete": false,\n    "confirmed": false,\n    "missing": [\n      "scenario.working_crs",\n      "instances.source",\n      "instances.format",\n      "instances.profile_column",\n      "antennas.<at_least_one>",\n      "interfaces.<at_least_one>",\n      "devices.<at_least_one>",\n      "metrics.<at_least_one>"\n    ],\n    "errors": [],\n    "next_action": "collect_missing_fields"\n  },\n  "scenario_snapshot": {\n    "scenario": {},\n    "antennas": {},\n    "interfaces": {},\n    "devices": {},\n    "metrics": {},\n    "instances": {}\n  }\n}'},
 {'role': 'assistant',
  'content': '{"action":"update_scenario","updates":{"scenario":{"working_crs":"EPSG:31982"}}}'},
 {'role': 'user',
  'content': 'USER MESSAGE\n============\n\nCrie a antena lte_sector_0. Ela é do tipo sector, modelo commscope_r

### Gerar e salvar `scenario.toml`

O arquivo será salvo no diretório do projeto:

`/home/jovyan/work/cisei_workspace/projects/<PROJECT_NAME>/scenario.toml`

A geração exige:

* `valid=True`;
* `complete=True`;
* `confirmed=True`.

O notebook não modifica o conteúdo do cenário antes da geração.


In [32]:
def gerar_e_salvar_toml():

    estado = obter_estado()

    # --------------------------------------------------------
    # VALIDAÇÃO
    # --------------------------------------------------------

    if not estado.get("valid"):
        raise RuntimeError(
            "O cenário ainda não está válido."
        )

    if not estado.get("complete"):
        raise RuntimeError(
            "O cenário ainda está incompleto."
        )

    if not estado.get("confirmed"):
        raise RuntimeError(
            "O cenário ainda não foi confirmado. "
            "Envie 'Confirmo o cenário.' pela conversa "
            "antes de gerar o TOML."
        )

    # --------------------------------------------------------
    # GERAÇÃO
    # --------------------------------------------------------

    toml_text = agent.scenario.generate_toml()

    # --------------------------------------------------------
    # SALVAMENTO
    # --------------------------------------------------------

    SCENARIO_PATH.write_text(
        toml_text,
        encoding="utf-8",
    )

    print("=" * 72)
    print("scenario.toml GERADO")
    print("=" * 72)
    print("Arquivo:", SCENARIO_PATH)
    print()
    print(toml_text)
    print("=" * 72)

    return toml_text

In [38]:
def mostrar_toml():

    if not SCENARIO_PATH.is_file():
        raise FileNotFoundError(
            f"scenario.toml não encontrado em: "
            f"{SCENARIO_PATH}"
        )

    toml_text = SCENARIO_PATH.read_text(
        encoding="utf-8"
    )

    print(toml_text)

    return toml_text

## 5. `nodes.csv`

O `nodes.csv` deve estar no mesmo diretório do projeto:

`/home/jovyan/work/cisei_workspace/projects/<PROJECT_NAME>/nodes.csv`

O notebook:

* verifica a existência do arquivo;
* lê o CSV;
* valida as colunas obrigatórias;
* converte campos numéricos necessários ao mapa;
* não inventa atributos;
* não altera o arquivo original.

In [34]:
import pandas as pd


REQUIRED_NODE_COLUMNS = [
    "site_id",
    "device_profile",
    "kind",
    "lat",
    "lon",
    "x",
    "y",
    "mount_height_m",
    "connected",
    "can_route",
    "rank",
]


def carregar_nodes():

    if not NODES_PATH.is_file():

        raise FileNotFoundError(
            f"nodes.csv não encontrado em: "
            f"{NODES_PATH}\n\n"
            "Coloque o CSV no diretório do projeto "
            "e execute novamente."
        )

    # --------------------------------------------------------
    # LEITURA
    # --------------------------------------------------------

    df = pd.read_csv(
        NODES_PATH
    )

    # --------------------------------------------------------
    # VALIDAÇÃO DO CONTRATO
    # --------------------------------------------------------

    missing = [
        column
        for column in REQUIRED_NODE_COLUMNS
        if column not in df.columns
    ]

    if missing:

        raise ValueError(
            "Colunas obrigatórias ausentes "
            f"em nodes.csv: {missing}"
        )

    # --------------------------------------------------------
    # CAMPOS NUMÉRICOS
    # --------------------------------------------------------

    for column in (
        "lat",
        "lon",
        "x",
        "y",
        "mount_height_m",
    ):

        df[column] = pd.to_numeric(
            df[column],
            errors="raise",
        )

    print("=" * 72)
    print("NODES CSV")
    print("=" * 72)
    print("Arquivo :", NODES_PATH)
    print("Linhas  :", len(df))
    print("Colunas :", list(df.columns))
    print("=" * 72)

    display(df)

    return df

## 6. MAPA `ipyleaflet`

O mapa usa:

* `lat` para latitude;
* `lon` para longitude.

Os campos `x` e `y` continuam disponíveis no DataFrame e não são reinterpretados pelo mapa.

Os nós são exibidos diretamente a partir do `nodes.csv`.



In [35]:
import html

from ipyleaflet import (
    Map,
    CircleMarker,
    Popup,
)

from ipywidgets import HTML


def criar_mapa_nodes(df):

    if df.empty:
        raise ValueError(
            "nodes.csv não contém linhas."
        )

    # --------------------------------------------------------
    # CENTRO INICIAL
    # --------------------------------------------------------

    center = (
        float(df["lat"].mean()),
        float(df["lon"].mean()),
    )

    mapa = Map(
        center=center,
        zoom=13,
        scroll_wheel_zoom=True,
        layout={
            "height": "700px"
        },
    )

    # --------------------------------------------------------
    # EXTENSÃO DOS DADOS
    # --------------------------------------------------------

    lat_min = float(
        df["lat"].min()
    )

    lat_max = float(
        df["lat"].max()
    )

    lon_min = float(
        df["lon"].min()
    )

    lon_max = float(
        df["lon"].max()
    )

    # --------------------------------------------------------
    # NÓS
    # --------------------------------------------------------

    for _, row in df.iterrows():

        kind = str(
            row["kind"]
        ).strip().lower()

        is_tower = (
            kind == "tower"
        )

        marker_color = (
            "red"
            if is_tower
            else "blue"
        )

        # ----------------------------------------------------
        # POPUP
        # ----------------------------------------------------

        popup_html = [
            "<b>Node</b><br>"
        ]

        for column in REQUIRED_NODE_COLUMNS:

            value = row[column]

            popup_html.append(
                f"<b>{html.escape(str(column))}</b>: "
                f"{html.escape(str(value))}<br>"
            )

        # ----------------------------------------------------
        # MARCADOR
        # ----------------------------------------------------

        marker = CircleMarker(
            location=(
                float(row["lat"]),
                float(row["lon"]),
            ),
            radius=(
                8
                if is_tower
                else 6
            ),
            color=marker_color,
            fill_color=marker_color,
            fill_opacity=0.8,
            weight=2,
        )

        marker.popup = Popup(
            child=HTML(
                value="".join(
                    popup_html
                )
            ),
            close_button=True,
            auto_close=True,
            close_on_escape_key=True,
        )

        mapa.add(
            marker
        )

    # --------------------------------------------------------
    # AJUSTAR EXTENSÃO
    # --------------------------------------------------------

    if (
        lat_min != lat_max
        or lon_min != lon_max
    ):

        mapa.fit_bounds(
            (
                (lat_min, lon_min),
                (lat_max, lon_max),
            )
        )

    return mapa

### Carregar o CSV e exibir o mapa

Execute esta célula **quando o `nodes.csv` já estiver no projeto**.


In [36]:
nodes = carregar_nodes()

mapa_nodes = criar_mapa_nodes(
    nodes
)

mapa_nodes

NODES CSV
Arquivo : /home/jovyan/work/cisei_workspace/projects/curitiba_lte/nodes.csv
Linhas  : 13
Colunas : ['site_id', 'device_profile', 'kind', 'lat', 'lon', 'x', 'y', 'mount_height_m', 'connected', 'can_route', 'rank']


,site_id,device_profile,kind,lat,lon,x,y,mount_height_m,connected,can_route,rank
0,copel,lte_leaf,field,-25.432731,-49.339207,667013.250467,7.186095e+06,7.0,False,False,inf
1,tenis,lte_leaf,field,-25.425357,-49.292122,671759.559806,7.186852e+06,7.0,False,False,inf
2,praca_29,lte_leaf,field,-25.427624,-49.285643,672408.065400,7.186593e+06,7.0,False,False,inf
3,torre,cell_sector_3x120,tower,-25.423532,-49.293992,671574.031187,7.187057e+06,100.0,True,True,0.0
4,woodland,lte_leaf,field,-25.426584,-49.330740,667873.384644,7.186766e+06,7.0,False,False,inf
5,deneka,lte_leaf,field,-25.423934,-49.325575,668396.573489,7.187053e+06,7.0,False,False,inf
6,cemiterio,lte_leaf,field,-25.420709,-49.313294,669636.488343,7.187394e+06,7.0,False,False,inf
7,saturno,lte_leaf,field,-25.427867,-49.343001,666638.395068,7.186639e+06,7.0,False,False,inf
8,decatlon,lte_leaf,field,-25.428662,-49.322405,668708.855680,7.186525e+06,7.0,False,False,inf
9,shopping,lte_leaf,field,-25.435108,-49.317711,669172.039893,7.185805e+06,7.0,False,False,inf


Map(center=[-25.427608973732575, -49.30821200743822], controls=(ZoomControl(options=['position', 'zoom_in_text…

## 7. AUDITORIA DA CONVERSA

Opcionalmente, podemos inspecionar todas as mensagens enviadas ao modelo.

Isso é útil durante os testes para verificar exatamente o contexto conversacional recebido pelo agente.


In [37]:
def mostrar_mensagens():

    for index, message in enumerate(
        agent.messages,
        start=1,
    ):

        print(
            f"\n{'=' * 72}"
        )

        print(
            f"MESSAGE {index}"
        )

        print(
            f"{'=' * 72}"
        )

        print(
            json.dumps(
                message,
                ensure_ascii=False,
                indent=2,
                default=str,
            )
        )


# Execute somente quando quiser auditar:
#
# mostrar_mensagens()

## 8. RESUMO DO PROJETO

Esta célula não executa planejamento.

Ela apenas mostra os artefatos associados ao projeto.


In [28]:
print("=" * 72)
print("CISEI SCENARIO PLANNER")
print("=" * 72)

print("Projeto       :", PROJECT_NAME)
print("Diretório     :", PROJECT_ROOT)
print("scenario.toml :", SCENARIO_PATH)
print("nodes.csv     :", NODES_PATH)
print("resultados    :", RESULTS_DIR)
print("relatórios    :", REPORTS_DIR)

print("=" * 72)

CISEI SCENARIO PLANNER
Projeto       : curitiba_lte
Diretório     : /home/jovyan/work/cisei_workspace/projects/curitiba_lte
scenario.toml : /home/jovyan/work/cisei_workspace/projects/curitiba_lte/scenario.toml
nodes.csv     : /home/jovyan/work/cisei_workspace/projects/curitiba_lte/nodes.csv
resultados    : /home/jovyan/work/cisei_workspace/projects/curitiba_lte/results
relatórios    : /home/jovyan/work/cisei_workspace/projects/curitiba_lte/reports
